In [31]:
import pandas as pd
import requests
import json
from dotenv import load_dotenv
import os
import pandas as pd
import time
from time import sleep
import ast
import itertools
load_dotenv()
API_KEY = os.getenv("API_KEY")
df1 = pd.read_csv('authors_data.csv')
df2 = pd.read_csv('works_ids.csv')


In [32]:
work_author_ids = df2["author_ids"].explode().unique()
existing_author_ids = df1["author_id"].explode().unique()
converted = [ast.literal_eval(x) for x in work_author_ids]
single_list_work_author_ids = [author for sublist in converted for author in sublist]
single_list_work_author_ids

['https://openalex.org/A5110328496',
 'https://openalex.org/A5077865016',
 'https://openalex.org/A5013693751',
 'https://openalex.org/A5064753440',
 'https://openalex.org/A5102851815',
 'https://openalex.org/A5030831197',
 'https://openalex.org/A5064753440',
 'https://openalex.org/A5083730471',
 'https://openalex.org/A5051345115',
 'https://openalex.org/A5017084936',
 'https://openalex.org/A5077865016',
 'https://openalex.org/A5058815871',
 'https://openalex.org/A5040821463',
 'https://openalex.org/A5000679279',
 'https://openalex.org/A5000679279',
 'https://openalex.org/A5040821463',
 'https://openalex.org/A5066696247',
 'https://openalex.org/A5000679279',
 'https://openalex.org/A5006700143',
 'https://openalex.org/A5081429241',
 'https://openalex.org/A5078651528',
 'https://openalex.org/A5000679279',
 'https://openalex.org/A5073348658',
 'https://openalex.org/A5010066039',
 'https://openalex.org/A5077865016',
 'https://openalex.org/A5038416473',
 'https://openalex.org/A5077865016',
 

In [ ]:
co_author_ids = []
for author_id in single_list_work_author_ids:
    if author_id not in existing_author_ids:
        co_author_ids.append(author_id)
co_author_ids = list(itertools.chain.from_iterable(co_author_ids))
total = len(co_author_ids)



907680


In [22]:
import requests
import pandas as pd
from itertools import islice
from time import sleep

rows4 = []
BASE_URL = "https://api.openalex.org/authors"

def chunked(iterable, size):
    it = iter(iterable)
    while True:
        chunk = list(islice(it, size))
        if not chunk:
            break
        yield chunk

session = requests.Session()

for batch_num, author_batch in enumerate(chunked(co_author_ids, 25), start=1):

    id_filter = "id:" + "|".join(author_batch)

    params = {
        "filter": id_filter,
        "select": "display_name,works_count,id,works_api_url,last_known_institutions,summary_stats",
        "per-page": 200,
        "api_key": API_KEY
    }

    try:
        response = session.get(BASE_URL, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.RequestException as e:
        print(f"Batch {batch_num} failed: {e}")
        continue

    results = data.get("results", [])
    print(f"Batch {batch_num}: fetched {len(results)} authors")

    for author_data in results:
        h_index = author_data.get("summary_stats", {}).get("h_index")
        institutions = author_data.get("last_known_institutions", [])
        country_code = institutions[0].get("country_code") if institutions else None

        rows4.append({
            "author_id": author_data.get("id"),
            "display_name": author_data.get("display_name"),
            "works_count": author_data.get("works_count"),
            "works_api_url": author_data.get("works_api_url"),
            "h_index": h_index,
            "country_code": country_code
        })

    sleep(0.2)  # small pause to avoid 429

df4 = pd.DataFrame(rows4)

print("Done:", len(df4))

Batch 1 failed: 400 Client Error: Bad Request for url: https://api.openalex.org/authors?filter=id%3Ah%7Ct%7Ct%7Cp%7Cs%7C%3A%7C%2F%7C%2F%7Co%7Cp%7Ce%7Cn%7Ca%7Cl%7Ce%7Cx%7C.%7Co%7Cr%7Cg%7C%2F%7CA%7C5%7C0%7C4&select=display_name%2Cworks_count%2Cid%2Cworks_api_url%2Clast_known_institutions%2Csummary_stats&per-page=200&api_key=VORYhfVyWtJjL5Og8auRQM
Batch 2 failed: 400 Client Error: Bad Request for url: https://api.openalex.org/authors?filter=id%3A9%7C1%7C8%7C4%7C0%7C5%7C1%7Ch%7Ct%7Ct%7Cp%7Cs%7C%3A%7C%2F%7C%2F%7Co%7Cp%7Ce%7Cn%7Ca%7Cl%7Ce%7Cx%7C.%7Co&select=display_name%2Cworks_count%2Cid%2Cworks_api_url%2Clast_known_institutions%2Csummary_stats&per-page=200&api_key=VORYhfVyWtJjL5Og8auRQM
Batch 3 failed: 400 Client Error: Bad Request for url: https://api.openalex.org/authors?filter=id%3Ar%7Cg%7C%2F%7CA%7C5%7C0%7C1%7C8%7C3%7C6%7C2%7C4%7C5%7C7%7Ch%7Ct%7Ct%7Cp%7Cs%7C%3A%7C%2F%7C%2F%7Co%7Cp%7Ce&select=display_name%2Cworks_count%2Cid%2Cworks_api_url%2Clast_known_institutions%2Csummary_stats&per-p

KeyboardInterrupt: 

In [10]:
df4.to_csv("co_authors_data.csv", index=False)